In [14]:
import os
import requests  
import json
import numpy as np
import tensorflow as tf
from tqdm import tqdm
import torch
import torch.nn as nn


In [3]:
def download_file(url, destination):
    try:
        # Send a GET request to download the file, disabling SSL verification
        response = requests.get(url, stream=True, verify=False)

        # Get the total file size from headers, defaulting to 0 if not present
        file_size = int(response.headers.get("content-length", 0))

        # Check if file exists and has the same size
        if os.path.exists(destination):
            file_size_local = os.path.getsize(destination)
            if file_size == file_size_local:
                print(f"File already exists and is up-to-date: {destination}")
                return

        # Define the block size for reading the file
        block_size = 1024  # 1 Kilobyte

        # Initialize the progress bar with total file size
        progress_bar_description = url.split("/")[-1]  # Extract filename from URL
        with tqdm(
            total=file_size, unit="iB", unit_scale=True, desc=progress_bar_description
        ) as progress_bar:
            # Open the destination file in binary write mode
            with open(destination, "wb") as file:
                # Iterate over the file data in chunks
                for chunk in response.iter_content(block_size):
                    progress_bar.update(len(chunk))  # Update progress bar
                    file.write(chunk)  # Write the chunk to the file

    except requests.exceptions.RequestException as e:
        print(f"Error downloading the file: {e}")
        print(f"Please check the URL: {url}")


In [4]:
def load_gpt2_params_from_tf_ckpt(ckpt_path, settings):
    # Initialize parameters dictionary with empty blocks for each layer
    params = {"blocks": [{} for _ in range(settings["n_layer"])]}

    # Iterate over each variable in the checkpoint
    for name, _ in tf.train.list_variables(ckpt_path):
        # Load the variable and remove singleton dimensions
        variable_array = np.squeeze(tf.train.load_variable(ckpt_path, name))

        # Process the variable name to extract relevant parts
        variable_name_parts = name.split("/")[1:]  # Skip the 'model/' prefix

        # Identify the target dictionary for the variable
        target_dict = params
        if variable_name_parts[0].startswith("h"):
            layer_number = int(variable_name_parts[0][1:])
            target_dict = params["blocks"][layer_number]

        # Recursively access or create nested dictionaries
        for key in variable_name_parts[1:-1]:
            target_dict = target_dict.setdefault(key, {})

        # Assign the variable array to the last key
        last_key = variable_name_parts[-1]
        target_dict[last_key] = variable_array

    return params


In [6]:
def download_and_load_gpt2(model_size, models_dir):
    # Validate model size
    allowed_sizes = ("124M", "355M", "774M", "1558M")
    if model_size not in allowed_sizes:
        raise ValueError(f"Model size not in {allowed_sizes}")

    # Define paths
    model_dir = os.path.join(models_dir, model_size)
    base_url = "https://openaipublic.blob.core.windows.net/gpt-2/models"
    filenames = [
        "checkpoint",
        "encoder.json",
        "hparams.json",
        "model.ckpt.data-00000-of-00001",
        "model.ckpt.index",
        "model.ckpt.meta",
        "vocab.bpe",
    ]

    # Download files
    os.makedirs(model_dir, exist_ok=True)
    for filename in filenames:
        file_url = os.path.join(base_url, model_size, filename)
        file_path = os.path.join(model_dir, filename)
        download_file(file_url, file_path)

    ## We have reached here until now ---> we have downloaded the files on our local machine.

    # Load settings and params
    tf_ckpt_path = tf.train.latest_checkpoint(model_dir)
    settings = json.load(open(os.path.join(model_dir, "hparams.json")))
    params = load_gpt2_params_from_tf_ckpt(tf_ckpt_path, settings)

    return settings, params


In [ ]:
settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")


In [9]:
print("Settings:", settings)
print("Parameter dictionary keys:", params.keys())


Settings: {'n_vocab': 50257, 'n_ctx': 1024, 'n_embd': 768, 'n_head': 12, 'n_layer': 12}
Parameter dictionary keys: dict_keys(['blocks', 'b', 'g', 'wpe', 'wte'])


In [10]:
print(params["wte"])
print("Token embedding weight tensor dimensions:", params["wte"].shape)


[[-0.11010301 -0.03926672  0.03310751 ... -0.1363697   0.01506208
   0.04531523]
 [ 0.04034033 -0.04861503  0.04624869 ...  0.08605453  0.00253983
   0.04318958]
 [-0.12746179  0.04793796  0.18410145 ...  0.08991534 -0.12972379
  -0.08785918]
 ...
 [-0.04453601 -0.05483596  0.01225674 ...  0.10435229  0.09783269
  -0.06952604]
 [ 0.1860082   0.01665728  0.04611587 ... -0.09625227  0.07847701
  -0.02245961]
 [ 0.05135201 -0.02768905  0.0499369  ...  0.00704835  0.15519823
   0.12067825]]
Token embedding weight tensor dimensions: (50257, 768)


In [11]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,  # Vocabulary size - used for tokenization
    "context_length": 1024,  # Context length - maximum number of tokens in a sequence for predicting of the next token
    "emb_dim": 768,  # Embedding dimension - Dimension of the token embeddings (contextual embeddings)
    "n_heads": 12,  # Number of attention heads - 12 pairs of attention heads in the multi-head attention mechanism
    "n_layers": 12,  # Number of layers - Number of transformer blocks in the model
    "drop_rate": 0.1,  # Dropout rate
    "qkv_bias": True,  # Query-Key-Value bias
}


In [12]:
# Define model configurations in a dictionary for compactness
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

# Copy the base configuration and update with specific model settings
model_name = "gpt2-small (124M)"  # Example model name
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
print(f"Configuration for {model_name}: {NEW_CONFIG}")


Configuration for gpt2-small (124M): {'vocab_size': 50257, 'context_length': 1024, 'emb_dim': 768, 'n_heads': 12, 'n_layers': 12, 'drop_rate': 0.1, 'qkv_bias': True}


In [15]:
class LayerNormalization(nn.Module):
    def __init__(self, embd_dim):
        super().__init__()
        # The variable eps is a small constant (epsilon) added to the variance to prevent division by zero during normalization.
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(embd_dim))
        self.shift = nn.Parameter(torch.zeros(embd_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        variance = x.var(dim=-1, keepdim=True, unbiased=False)
        normalized_output = (x - mean) / torch.sqrt(variance + self.eps)
        return self.scale * normalized_output + self.shift


In [16]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return (
            0.5
            * x
            * (
                1
                + torch.tanh(
                    torch.sqrt(torch.tensor(2.0 / torch.pi))
                    * (x + 0.044715 * torch.pow(x, 3))
                )
            )
        )


class FeedForwardLayer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(
                config["emb_dim"], 4 * config["emb_dim"]
            ),  # First linear layer expands the embedding dimension by a factor of 4
            GELU(),  # GELU activation function
            nn.Linear(
                4 * config["emb_dim"], config["emb_dim"]
            ),  # Second linear layer projects back to the original embedding dimension
        )

    def forward(self, x):
        return self.layers(x)


In [17]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_len, dropout, no_of_heads, qkv_bias=False):
        super().__init__()
        assert d_out % no_of_heads == 0, "d_out must be divisible by no_of_heads"

        self.no_of_heads = no_of_heads
        self.d_out = d_out
        self.d_out_per_head = d_out // no_of_heads
        # since , d_out = no_of_heads * d_out_per_head , so if d_out = d_in for gpt like models and for eg = 6 and no_of_heads = 2 , then d_out_per_head = 3

        # Initializing the weight matrices for query, key and value for multi-head attention
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_len, context_len), diagonal=1)
        )

    def forward(self, x):
        batch_size, num_tokens, d_in = x.shape
        # print(
        #     f"Input vector Dimensions Info - Batch Size : {batch_size}, Number of Tokens : {num_tokens}, Input Dimension(dimension of vector/token embedding) : {d_in}"
        # )

        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # The shape of K,Q,V Should be (batch_size, num_tokens, d_out)
        # print("Keys Shape : ", keys.shape)
        # print("Queries Shape : ", queries.shape)
        # print("Values Shape : ", values.shape)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (batch_size, num_tokens, d_out) -> (batch_size, num_tokens, num_heads, d_out_per_head)
        # as d_out = no_of_heads * d_out_per_head

        keys = keys.view(batch_size, num_tokens, self.no_of_heads, self.d_out_per_head)
        queries = queries.view(
            batch_size, num_tokens, self.no_of_heads, self.d_out_per_head
        )
        values = values.view(
            batch_size, num_tokens, self.no_of_heads, self.d_out_per_head
        )
        # print(
        #     "Keys Shape after adding num_heads dimension : [batch_size, num_tokens, num_heads, d_out_per_head] ",
        #     keys.shape,
        # )
        # print(
        #     "Queries Shape after adding num_heads dimension : [batch_size, num_tokens, num_heads, d_out_per_head] ",
        #     queries.shape,
        # )
        # print(
        #     "Values Shape after adding num_heads dimension : [batch_size, num_tokens, num_heads, d_out_per_head] ",
        #     values.shape,
        # )

        # Transpose: (batch_size, num_tokens, no_of_heads, d_out_per_head) -> (batch_size, no_of_heads, num_tokens, d_out_per_head) , we do this because we want to perform the dot product between the query and key vectors for each head separately (parallel computation) so we group on basis on no of heads, and then combine the results to get the final context vector.
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # print(
        #     "Keys Shape after Transpose : [batch_size, no_of_heads, num_tokens, d_out_per_head] ",
        #     keys.shape,
        # )
        # print(
        #     "Queries Shape after Transpose : [batch_size, no_of_heads, num_tokens, d_out_per_head] ",
        #     queries.shape,
        # )
        # print(
        #     "Values Shape after Transpose : [batch_size, no_of_heads, num_tokens, d_out_per_head] ",
        #     values.shape,
        # )

        # Now, the shape of the keys, queries, and values will be (batch_size, no_of_heads, num_tokens, d_out_per_head)
        # Now, we can compute the attention scores for each head separately using batch matrix multiplication. The attention scores will have the shape (batch_size, no_of_heads, num_tokens, num_tokens), where each head computes the attention scores for all tokens in the sequence.

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)
        # print(
        #     "Attention Scores Shape : [batch_size, no_of_heads, num_tokens, num_tokens] ",
        #     attn_scores.shape,
        # )

        # queries = (batch_size, no_of_heads, num_tokens, d_out_per_head) = (2, 2, 3, 3)
        # keys = (batch_size, no_of_heads, num_tokens, d_out_per_head) = (2, 2, 3, 3)
        # keys.transpose(2, 3) = (batch_size, no_of_heads, d_out_per_head, num_tokens) = (2, 2, 3, 3)

        # attn_scores = (batch_size, no_of_heads, num_tokens, num_tokens) = (2, 2, 3, 3)

        # Apply the causal mask to prevent attending to future tokens , Original mask truncated to the number of tokens
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores = attn_scores.masked_fill(mask_bool, -torch.inf)

        # Apply softmax to get the attention weights and scale by the square root of the dimension of the key vectors
        attn_weights = torch.softmax(attn_scores / self.d_out_per_head**0.5, dim=-1)
        # or can be : attn_Weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5 , dim=-1)

        # print(
        #     "Attention Weights Shape : [batch_size, no_of_heads, num_tokens, num_tokens] ",
        #     attn_weights.shape,
        # )

        # Apply dropout to the attention weights
        attn_weights = self.dropout(attn_weights)

        # Compute the context vectors by multiplying the attention weights with the value vectors
        context_vectors = (attn_weights @ values).transpose(1, 2)
        # print(
        #     "Context Vectors Shape after Transpose : [batch_size, num_tokens, no_of_heads, d_out_per_head] ",
        #     context_vectors.shape,
        # )

        # why transpose(1, 2) here? Because we want to bring the no_of_heads dimension back to the last dimension so that we can concatenate the context vectors from all heads. After this operation, the shape of context_vectors will be (batch_size, num_tokens, no_of_heads, d_out_per_head).

        # attn_weights = (batch_size, no_of_heads, num_tokens, num_tokens) = (2, 2, 3, 3)
        # values       = (batch_size, no_of_heads, num_tokens, d_out_per_head) = (2, 2, 3, 3)

        # initial_context_vector = (attn_weights @ values) = (batch_size, no_of_heads, num_tokens, d_out_per_head) = (2, 2, 3, 3)
        # After transpose(1, 2), context_vectors           = (batch_size, num_tokens, no_of_heads, d_out_per_head) = (2, 3, 2, 3)

        # Back to the original shape of (batch_size, num_tokens, d_out) by concatenating the context vectors from all heads along the last dimension
        context_vectors = context_vectors.contiguous().view(
            batch_size, num_tokens, self.d_out
        )
        # print(
        #     "Context Vectors Shape after Concatenation : [batch_size, num_tokens, d_out] ",
        #     context_vectors.shape,
        # )

        # contiguous() is used to ensure that the tensor is stored in a contiguous block of memory, which is required for the view operation to work correctly. The view operation reshapes the tensor to have the shape (batch_size, num_tokens, d_out), where d_out = no_of_heads * d_out_per_head.
        # so the final shape of context_vectors will be (batch_size, num_tokens, d_out) = (2, 3, 6)

        # optional projection layer to combine the outputs from all heads into a single output vector of dimension d_out
        context_vectors = self.out_proj(context_vectors)
        # print(
        #     "Context Vectors Shape after Out Projection : [batch_size, num_tokens, d_out] ",
        #     context_vectors.shape,
        # )

        return context_vectors


In [18]:
class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attention_block = MultiHeadAttention(
            d_in=config["emb_dim"],
            d_out=config["emb_dim"],
            context_len=config["context_length"],
            no_of_heads=config["n_heads"],
            dropout=config["drop_rate"],
            qkv_bias=config["qkv_bias"],
        )

        # Intializing the feed forward block, layer normalization layers and dropout layer for the transformer block.
        self.feed_forward_block = FeedForwardLayer(config)
        self.layer_norm1 = LayerNormalization(config["emb_dim"])
        self.layer_norm2 = LayerNormalization(config["emb_dim"])
        self.dropout = nn.Dropout(config["drop_rate"])

    def forward(self, x):
        # The forward method defines how the input data flows through the transformer block.

        # Intially the shortcut connection is set to the input x, which will be used later for residual connections.
        shortcut = x
        # Apply layer normalization to the input x
        x = self.layer_norm1(x)
        # Pass the normalized input through the multi-head - shape - [batch_size, num_tokens, d_out=emb_dim]
        x = self.attention_block(x)
        # Apply dropout to the output of the attention block
        x = self.dropout(x)
        # Add the shortcut connection to the output of the attention block
        x = x + shortcut
        # The above x is the combined output of the attention block and the shortcut connection, Acting as the input to the feed forward block.

        # Shortcut connection for the feed forward block is set to the current x, which will be used later for residual connections.
        shortcut = x
        # Apply layer normalization to the current x
        x = self.layer_norm2(x)
        # Pass the normalized input through the feed forward block - shape - [batch_size, num_tokens, d_out=emb_dim]
        x = self.feed_forward_block(x)
        # Apply dropout to the output of the feed forward block
        x = self.dropout(x)
        # Add the shortcut connection to the output of the feed forward block
        x = x + shortcut

        # The final output of the transformer block is returned after passing through the attention and feed forward blocks with residual connections.
        return x


In [19]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # Shape = [vocab_size, emb_dim] : [50257, 768]
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])

        # Shape = [context_length, emb_dim] : [1024, 768] , why only context_length? Because we only use the current context length (1024) tokens to predict the next token, we only need to learn position embeddings for the first 1024 positions. Thus, the position embedding matrix is of size (1024, 768).
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])

        self.dropout = nn.Dropout(cfg["drop_rate"])

        # Intialize the 12 transformer blocks, each block consists of a multi-head attention layer followed by a feed-forward layer, with residual connections and layer normalization applied at each step. The number of layers is specified in the configuration (cfg["n_layers"]).
        self.transformer_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        # Layer normalization is applied to the final hidden states of the model before they are passed to the output projection layer. This helps stabilize training and improve convergence. It is applied across the feature dimension (emb_dim) [row wise] for each token in the sequence, ensuring that the activations have a mean of 0 and a standard deviation of 1. This normalization step helps prevent issues like vanishing or exploding gradients during training.
        self.final_norm = LayerNormalization(cfg["emb_dim"])

        # The output projection layer is a linear layer that maps the final hidden states of the model to the vocabulary size. This is used to generate logits for each token in the vocabulary, which can then be used to predict the next token in a sequence. The bias term is set to False because we want the model to learn a direct mapping from the hidden states to the logits without any additional bias.
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.dropout(x)
        x = self.transformer_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits


In [20]:
gpt = GPTModel(NEW_CONFIG)
gpt.eval()


GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (dropout): Dropout(p=0.1, inplace=False)
  (transformer_blocks): Sequential(
    (0): TransformerBlock(
      (attention_block): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (feed_forward_block): FeedForwardLayer(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (layer_norm1): LayerNormalization()
      (layer_norm2): LayerNormalization()
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (a